# 05 — Understand and review the self-improvement loop

The required feedback loop is implemented in `improve.py`; this notebook is its presentation/review view, not the loop engine. During core development, one command evaluated train data, diagnosed failures, selected an untried configuration change, evaluated it, and accepted or rejected it using declared rules.

The separate controlled hosted/local comparisons use `evaluate_llm.py` and cached outputs. This notebook counts all unique iterations across the three logs. It does not call a model, run a new trial, or evaluate test performance.

The ten-iteration budget is now complete. The core command prevents further proposals during the LLM comparison phase; use the saved review notebooks for presentation.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from improve import read_log, read_json, resume_state
from retrieval import BM25Retriever, load_inputs

historical = read_log(ROOT / 'experiments/log.jsonl')
history = read_log(ROOT / 'experiments/loop_log.jsonl')
assert history, 'Run python improve.py once before reviewing results.'
current, tried_ids, tried_configs, next_iteration = resume_state(
    read_json(ROOT / 'configs/bm25_baseline.json'), historical, history)
llm_history = read_log(ROOT / 'experiments/llm_log.jsonl')
completed = {r['iteration'] for r in historical + history + llm_history}
limit = read_json(ROOT / 'experiments/limits.json')['max_iterations']
print('Distinct completed iterations:', len(completed))
print('Required minimum beyond baseline: 5')
print('Current selected approach:', current['name'])
print('Remaining allowed iterations:', max(0, limit - len(completed)))
print('LLM comparisons:', sorted({r['iteration'] for r in llm_history}))
display(pd.DataFrame(read_json(ROOT / 'experiments/proposals.json')))

Distinct completed iterations: 10
Required minimum beyond baseline: 5
Current selected approach: bm25_focused_stemmed
Remaining allowed iterations: 0
LLM comparisons: [9, 10]


,id,name,trigger,requires,changes,reason
0,english_stemming_on_focused_pipeline,bm25_focused_stemmed,right_document_wrong_section,"{'query_mode': 'document_then_clause', 'clause...",{'stemming': 'english'},Normalize word inflections in indexed text and...
1,focused_clause_with_fallback,bm25_focused_clause_fallback,right_document_wrong_section,{'query_mode': 'document_then_clause'},"{'clause_query_cleanup': True, 'clause_cleanup...",Use focused clause terms when they produce mat...
2,clause_query_cleanup,bm25_focused_clause,right_document_wrong_section,{'query_mode': 'document_then_clause'},{'clause_query_cleanup': True},Remove function words and generic contract-que...
3,document_then_clause,bm25_document_then_clause,right_document_wrong_section,NaN,{'query_mode': 'document_then_clause'},"Use the full question to identify a contract, ..."
4,smaller_chunks,bm25_chunks_500,right_document_wrong_section,NaN,{'chunk_size': 500},Shorter passages may separate relevant clauses...
5,larger_chunks,bm25_chunks_1500,partial_coverage,NaN,{'chunk_size': 1500},Longer passages may cover answer spans split a...
6,less_length_normalization,bm25_b_030,right_document_wrong_section,NaN,{'b': 0.3},Less length normalization may reduce ranking b...
7,lower_term_saturation,bm25_k1_080,right_document_wrong_section,NaN,{'k1': 0.8},Earlier term-frequency saturation may reduce t...
8,positive_score_threshold,bm25_threshold_50,false_returns,NaN,{'min_score': 50.0},A higher score threshold may reduce false retu...


## Latest core-loop decision and baseline comparison

Acceptance compares the candidate with the current accepted configuration. Every result also retains the original baseline. The common-word experiment remains recorded separately as iteration 1; reruns of that same proposal are not new iterations.

Our primary objective is recall@5. Precision, both abstention rates, and local p95 search latency are guardrails. The baseline's zero negative abstention means the negative guardrail is initially weak; passing it does not establish safe answerability detection.

The core decision below is iteration 8. Hosted/local iterations 9 and 10 are reviewed in notebooks 10 and 11.

In [2]:
latest = history[-1]
print('Iteration:', latest['iteration'])
print('Proposal:', latest['proposal']['id'])
print('Reason:', latest['proposal']['reason'])
print('Diagnosis:', latest['diagnosis_before'])
print('Accepted:', latest['decision']['accepted'])
display(pd.Series(latest['decision']['checks'], name='Passed'))
rows = []
for label, result in latest['metrics'].items():
    at5 = next(m for m in result['retrieval'] if m['k'] == 5)
    rows.append(dict(approach=label, precision_at_5=at5['precision'], recall_at_5=at5['recall'],
                     false_abstention=result['false_abstention'], correct_abstention=result['correct_abstention'],
                     p95_ms=result['latency_ms']['p95']))
display(pd.DataFrame(rows))
print('Selected configuration:')
display(pd.Series(latest['selected_config']))

Iteration: 8
Proposal: english_stemming_on_focused_pipeline
Reason: Normalize word inflections in indexed text and search tokens, building on both accepted query policies without changing source text or offsets.
Diagnosis: {'complete_coverage': 57, 'right_document_wrong_section': 50, 'partial_coverage': 33, 'false_returns': 23}
Accepted: True


recall_gain                True
precision                  True
answerable_abstention      True
unanswerable_abstention    True
latency                    True
Name: Passed, dtype: bool

,approach,precision_at_5,recall_at_5,false_abstention,correct_abstention,p95_ms
0,baseline,0.017433,0.181647,0.0,0.0,8.47192
1,reference,0.062782,0.535021,0.0,0.0,10.98969
2,candidate,0.078798,0.670075,0.0,0.0,10.49893


Selected configuration:


name                       bm25_focused_stemmed
chunk_size                                 1000
chunk_overlap                               200
k1                                          1.5
b                                          0.75
min_score                                   0.0
k_values                          [1, 3, 5, 10]
query_mode                 document_then_clause
clause_query_cleanup                       True
clause_cleanup_fallback                    True
stemming                                english
dtype: object

## Run either saved configuration

Edit `which` to `reference_config`, `candidate_config`, or `selected_config`. Even a rejected approach can be loaded and used. The code keeps settings separate from retrieval so they can be changed during an interview.

In [3]:
which = 'candidate_config'
_, documents, _ = load_inputs(ROOT)
model = BM25Retriever(documents, latest[which])
question = 'Under the Fox and Glu Mobile wireless content license, when does the term expire?'
hits = model.search(question, k=5)
if hits:
    display(pd.DataFrame(hits)[['score', 'file_path', 'start', 'end', 'text']])
else:
    print('No relevant passage found.')

,score,file_path,start,end,text
0,49.889711,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,0,1000,"AMENDMENT NO. 3\n\nDated as of February 19, 20..."
1,33.883776,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,800,1800,"dom of Heaven, Mr. and Mrs. Smith, In Her Shoe..."
2,29.672171,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,1600,1902,"x Mobile Entertainment, Inc. (""Fox"")\n\n\n\nBy..."
3,18.725606,cuad/KINGPHARMACEUTICALSINC_08_09_2006-EX-10.1...,63200,64200,ssion pursuant to the Company's application re...
4,18.615700,cuad/ALCOSTORESINC_12_14_2005-EX-10.26-AGENCY ...,5600,6600,"to any documentation, technical or confidenti..."


## Explain this step

"The script closes the feedback loop: measure, diagnose, propose, evaluate, accept or reject, and log. The notebook makes those decisions visible so we can explain them. It does not itself train a model or rewrite code."

The loop is required by the assignment; using an ipynb file is our presentation choice. The minimum is five distinct improvements beyond baseline, and our ten-iteration budget is complete. Accepted and rejected trials both count; replaying a notebook does not create a new proposal.

Core trials are recorded in `experiments/log.jsonl` and `experiments/loop_log.jsonl`; the two LLM comparisons are in `experiments/llm_log.jsonl`. Both use train-only evaluation and fixed acceptance criteria. The web/search CLI are implemented; final reporting and held-out evaluation remain pending.